# Inspección visual de ROI (cascada binaria → multiclase) — V3

Notebook derivado de `train_spine_cascade_binary_to_thoracolumbar_v3.ipynb` para **revisar sistemáticamente** las ROI generadas por la etapa binaria (`thoracolumbar_core_binary_rois.csv`) y su relación con las **vértebras de interés** (p. ej. tramo torácico medio T9–T11 y L5).

**Objetivo:** detectar recortes mal centrados, cortes en bordes o desalineaciones que expliquen regresiones por clase sin reentrenar modelos.


## Requisitos

- `analysis_outputs_v3/training_runs_cascade_v3/thoracolumbar_core_split_train_val_test.csv`
- `analysis_outputs_v3/training_runs_cascade_v3/thoracolumbar_core_binary_rois.csv`
- `Scoliosis_Dataset_V3/Scoliosis_Dataset/diccionario_etiquetas_T1_T12_L1_L5.json`

No hace falta cargar pesos PyTorch: solo imágenes y máscaras en disco.


In [ ]:
from __future__ import annotations

import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def _resolve_maia_project_root(marker: Path) -> Path:
    """Raiz del repo (carpeta que contiene marker). Local, subcarpetas, Colab+Drive."""
    env = os.environ.get('MAIA_PROJECT_ROOT', '').strip()
    if env:
        p = Path(env).expanduser().resolve()
        if (p / marker).exists():
            return p
    cwd = Path.cwd().resolve()
    for cand in [cwd, *cwd.parents]:
        if (cand / marker).exists():
            return cand
    drive_nb = Path('/content/drive/MyDrive/Colab Notebooks')
    if drive_nb.is_dir():
        direct = drive_nb / 'MAIA-PROYECTO'
        if (direct / marker).exists():
            return direct.resolve()
        for child in drive_nb.iterdir():
            try:
                if child.is_dir() and (child / marker).exists():
                    return child.resolve()
            except OSError:
                continue
    drive_maia = Path('/content/drive/MyDrive/MAIA-PROYECTO')
    if (drive_maia / marker).exists():
        return drive_maia.resolve()
    raise FileNotFoundError(
        f"No se encontro {marker.as_posix()}. Opciones: (1) %cd a la raiz del repo; "
        f"(2) os.environ['MAIA_PROJECT_ROOT'] = r'.../MAIA-PROYECTO'. cwd={cwd}"
    )


ROOT = _resolve_maia_project_root(Path('data') / 'Scoliosis_Dataset')

DATASET_DIR = ROOT / 'data' / 'Scoliosis_Dataset'
LABELS_DICT_PATH = DATASET_DIR / 'diccionario_etiquetas_T1_T12_L1_L5.json'
OUTPUT_DIR = ROOT / 'outputs' / 'analysis_outputs_v3' / 'training_runs_cascade_v3'
MULTICLASS_SUBSET = 'core'
IGNORE_INDEX = 255
ROI_PAD_X = 18
ROI_PAD_Y = 30
MIN_FOREGROUND_PIXELS = 24

TARGET_LABELS = [f'T{i}' for i in range(1, 13)] + [f'L{i}' for i in range(1, 6)]
LABEL_TO_TRAIN_ID = {name: idx + 1 for idx, name in enumerate(TARGET_LABELS)}

with open(LABELS_DICT_PATH, 'r', encoding='utf-8') as f:
    labels_dict = json.load(f)
multiclass_key = 'multiclass_id_png' if 'multiclass_id_png' in labels_dict else 'mascara_multiclase_id_png'
multiclass_map = {int(k): v for k, v in labels_dict[multiclass_key].items()}
label_to_id = {label: class_id for class_id, label in multiclass_map.items()}
target_id_to_train_id = {label_to_id[label]: idx + 1 for idx, label in enumerate(TARGET_LABELS)}

print('ROOT:', ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)


In [ ]:
split_path = OUTPUT_DIR / f'thoracolumbar_{MULTICLASS_SUBSET}_split_train_val_test.csv'
roi_path = OUTPUT_DIR / f'thoracolumbar_{MULTICLASS_SUBSET}_binary_rois.csv'

multiclass_splits_df = pd.read_csv(split_path)
roi_df = pd.read_csv(roi_path)

merged = multiclass_splits_df.merge(roi_df, on='unique_sample_id', how='inner', suffixes=('', '_roi'))
assert len(merged) == len(roi_df) == len(multiclass_splits_df), 'Filas inconsistentes entre split y ROI'

print('Muestras:', len(merged))
display(merged[['unique_sample_id', 'partition', 'roi_source', 'bbox_width', 'bbox_height']].head())
display(merged['roi_source'].value_counts())


In [ ]:
def read_gray(path: str) -> np.ndarray:
    return np.array(Image.open(path).convert('L'))


def resize_mask(arr: np.ndarray, size: tuple[int, int]) -> np.ndarray:
    return np.array(Image.fromarray(arr.astype(np.uint8)).resize((size[1], size[0]), resample=Image.NEAREST))


def build_binary_mask(path: str, size: tuple[int, int] | None = None) -> np.ndarray:
    mask = read_gray(path)
    mask = (mask >= 127).astype(np.uint8)
    if size is not None:
        mask = resize_mask(mask, size)
    return mask


def build_multiclass_mask(path: str, size: tuple[int, int] | None = None) -> np.ndarray:
    raw = np.array(Image.open(path), dtype=np.int32)
    out = np.zeros_like(raw, dtype=np.uint8)
    out[raw != 0] = IGNORE_INDEX
    for old_id, new_id in target_id_to_train_id.items():
        out[raw == old_id] = new_id
    if size is not None:
        out = resize_mask(out, size)
    return out


def bbox_from_mask(mask: np.ndarray, min_foreground_pixels: int = 24) -> tuple[int, int, int, int] | None:
    ys, xs = np.where(mask > 0)
    if len(xs) < min_foreground_pixels:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1


def clamp_bbox(bbox: tuple[int, int, int, int], image_shape: tuple[int, int]) -> tuple[int, int, int, int]:
    h, w = image_shape
    x0, y0, x1, y1 = bbox
    x0 = max(0, min(x0, w - 1))
    y0 = max(0, min(y0, h - 1))
    x1 = max(x0 + 1, min(x1, w))
    y1 = max(y0 + 1, min(y1, h))
    return x0, y0, x1, y1


def expand_bbox(
    bbox: tuple[int, int, int, int],
    image_shape: tuple[int, int],
    pad_x: int = 18,
    pad_y: int = 30,
) -> tuple[int, int, int, int]:
    x0, y0, x1, y1 = bbox
    return clamp_bbox((x0 - pad_x, y0 - pad_y, x1 + pad_x, y1 + pad_y), image_shape)


def crop_array(arr: np.ndarray, bbox: tuple[int, int, int, int]) -> np.ndarray:
    x0, y0, x1, y1 = bbox
    return arr[y0:y1, x0:x1]


def extents_for_train_id(mask: np.ndarray, train_id: int) -> tuple[int, int, int, int] | None:
    ys, xs = np.where(mask == train_id)
    if len(xs) == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1


def gt_expanded_bbox_from_binary(row: pd.Series) -> tuple[int, int, int, int]:
    """Misma lógica que ROI de entrenamiento multiclase con máscara binaria GT."""
    binary_raw = build_binary_mask(row['binary_mask_path_abs'], size=None)
    sh = binary_raw.shape
    bb = bbox_from_mask(binary_raw, min_foreground_pixels=MIN_FOREGROUND_PIXELS)
    if bb is None:
        h, w = sh
        return 0, 0, w, h
    return expand_bbox(bb, image_shape=sh, pad_x=ROI_PAD_X, pad_y=ROI_PAD_Y)


## Visualización de ROI y vértebras a analizar

Por cada muestra se muestra:

1. **Radiografía completa** con rectángulo de la ROI **predicha** (verde) y, en trazo discontinuo, la ROI que se obtendría desde la **máscara binaria GT** (cian), con la misma expansión `ROI_PAD_*` que en entrenamiento.
2. **Marcas horizontales** por vértebra en `FOCUS_VERTEBRAE`: línea en la **y central** del hueso en coordenadas de imagen completa (útil para ver si T9–T11 quedan cerca del borde del recorte).
3. **Recorte multiclase** según la ROI predicha, coloreado por etiqueta (misma codificación que el entrenamiento).

Ajusta `PARTITION_FILTER`, `N_SAMPLES`, `FOCUS_VERTEBRAE` y `SAMPLE_IDS` en la celda siguiente.


In [ ]:
# --- Parámetros de inspección ---
FOCUS_VERTEBRAE = ['T9', 'T10', 'T11', 'T7', 'T12', 'L5', 'T1']
PARTITION_FILTER = 'test'   # 'train' | 'val' | 'test' | None (todas)
N_SAMPLES = 12
RANDOM_SEED = 42
# Lista opcional de unique_sample_id para forzar casos concretos (si no vacía, ignora N_SAMPLES aleatorio)
SAMPLE_IDS: list[str] = []

CMAP = plt.cm.nipy_spectral
COLORS_LINE = plt.cm.tab10(np.linspace(0, 1, max(len(FOCUS_VERTEBRAE), 3)))


def plot_roi_inspection_grid(
    table: pd.DataFrame,
    focus: list[str],
    partition: str | None,
    n_samples: int,
    seed: int,
    sample_ids: list[str],
) -> None:
    work = table.copy()
    if partition is not None:
        work = work.loc[work['partition'] == partition]
    if sample_ids:
        use = work.loc[work['unique_sample_id'].isin(sample_ids)]
        miss = set(sample_ids) - set(use['unique_sample_id'])
        if miss:
            print('Advertencia: no encontrados en la tabla filtrada:', miss)
    else:
        work = work.sample(n=min(n_samples, len(work)), random_state=seed)
        use = work

    if use.empty:
        print('Sin filas para mostrar.')
        return

    n = len(use)
    fig, axes = plt.subplots(n, 2, figsize=(14, 4 * n))
    if n == 1:
        axes = np.array([axes])

    for r, (_, row) in enumerate(use.iterrows()):
        img = read_gray(row['radiograph_path_abs'])
        mcls = build_multiclass_mask(row['multiclass_mask_path_abs'], size=None)
        assert img.shape == mcls.shape

        pred_x0 = int(row['bbox_x0'])
        pred_y0 = int(row['bbox_y0'])
        pred_x1 = int(row['bbox_x1'])
        pred_y1 = int(row['bbox_y1'])

        gt_bb = gt_expanded_bbox_from_binary(row)
        gx0, gy0, gx1, gy1 = gt_bb

        ax0 = axes[r, 0]
        ax0.imshow(img, cmap='gray', aspect='auto')
        ax0.add_patch(
            Rectangle(
                (pred_x0, pred_y0),
                pred_x1 - pred_x0,
                pred_y1 - pred_y0,
                fill=False,
                edgecolor='lime',
                linewidth=2.2,
                label='ROI pred_binary',
            )
        )
        ax0.add_patch(
            Rectangle(
                (gx0, gy0),
                gx1 - gx0,
                gy1 - gy0,
                fill=False,
                edgecolor='cyan',
                linewidth=1.8,
                linestyle='--',
                label='ROI desde binaria GT',
            )
        )

        for i, name in enumerate(focus):
            tid = LABEL_TO_TRAIN_ID.get(name)
            if tid is None:
                continue
            ex = extents_for_train_id(mcls, tid)
            if ex is None:
                continue
            _, y0b, _, y1b = ex
            yc = (y0b + y1b) / 2
            ax0.axhline(yc, color=COLORS_LINE[i % len(COLORS_LINE)], linewidth=1.0, alpha=0.9)
            ax0.text(
                8,
                yc,
                f' {name}',
                color=COLORS_LINE[i % len(COLORS_LINE)],
                fontsize=9,
                va='center',
                bbox=dict(boxstyle='round,pad=0.15', facecolor='black', alpha=0.35),
            )

        ax0.set_title(
            f"{row['unique_sample_id']}  |  {row['partition']}  |  roi={row.get('roi_source', '')}",
            fontsize=10,
        )
        ax0.legend(loc='upper right', fontsize=8)
        ax0.axis('off')

        crop = crop_array(img, (pred_x0, pred_y0, pred_x1, pred_y1))
        mcrop = crop_array(mcls, (pred_x0, pred_y0, pred_x1, pred_y1))
        vis = mcrop.astype(np.float32).copy()
        vis[vis == IGNORE_INDEX] = np.nan

        ax1 = axes[r, 1]
        ax1.imshow(crop, cmap='gray', aspect='auto')
        ovl = ax1.imshow(vis, cmap=CMAP, vmin=0, vmax=len(TARGET_LABELS), alpha=0.45, aspect='auto')
        ax1.set_title('Recorte según ROI predicha (overlay multiclase)', fontsize=10)
        ax1.axis('off')

    plt.tight_layout()
    plt.show()


plot_roi_inspection_grid(
    merged,
    focus=FOCUS_VERTEBRAE,
    partition=PARTITION_FILTER,
    n_samples=N_SAMPLES,
    seed=RANDOM_SEED,
    sample_ids=SAMPLE_IDS,
)
